# 에피소딕 메모리를 활용한 디버깅 어시스턴트

## 개요

이 노트북은 **AgentCore 에피소딕 메모리**와 리플렉션을 사용하여 지능형 **디버깅 어시스턴트**를 구축하는 방법을 보여줍니다. 에이전트는 과거 디버깅 세션에서 학습하고 이전 경험을 기반으로 컨텍스트 인식 가이드를 제공합니다.

### 에피소딕 메모리란?

**에피소딕 메모리**는 구조화된 컨텍스트와 함께 완전한 상호작용 시퀀스(에피소드)를 캡처합니다. 고립된 사실을 저장하는 시맨틱 메모리와 달리, 에피소딕 메모리는 다음을 보존합니다:
- **전체 대화 흐름**: 문제 진술에서 해결까지의 완전한 디버깅 세션
- **시간적 컨텍스트**: 수행된 작업의 순서와 타이밍
- **결과**: 디버깅 시도의 성공 또는 실패 여부
- **구조화된 턴**: 사고, 행동, 관찰이 포함된 개별 단계

![에피소딕 메모리](./episodic_memory.png)

### 리플렉션이란?

**리플렉션**은 여러 에피소드에서 자동으로 추출된 합성된 인사이트입니다. 다음을 제공합니다:
- **패턴 인식**: 유사한 에피소드에서의 공통 문제와 해결책
- **모범 사례**: 성공적인 디버깅 세션에서 잘 작동한 전략
- **일반적인 함정**: 실패한 시도에서 피해야 할 실수
- **전략적 가이드**: 유사한 문제에 접근하기 위한 고수준 조언

**출력 구조:**
- **에피소드**: `debugging/{actorId}/sessions/{sessionId}`에 저장 - 전체 대화 추적
- **리플렉션**: `debugging/{actorId}`에 저장 - 여러 에피소드에서 합성된 지식

### 에피소딕 메모리를 사용하는 경우

에피소딕 메모리는 다음과 같은 경우에 사용합니다:
1. **순차적 컨텍스트가 중요한 경우**: 작업의 순서와 결과가 중요한 경우 (예: 디버깅 워크플로, 문제 해결 절차)
2. **경험에서 학습하는 경우**: 과거 성공과 실패를 분석하여 에이전트가 개선되기를 원하는 경우
3. **프로세스 검색이 필요한 경우**: 사용자가 "지난번에 X를 어떻게 해결했지?" 또는 "수행한 정확한 단계를 보여줘"를 기억해야 하는 경우

### 튜토리얼 세부사항

| 정보 | 세부사항 |
|:------------|:--------|
| 튜토리얼 유형 | 리플렉션이 포함된 에피소딕 메모리 |
| 에이전트 유형 | 디버깅 어시스턴트 |
| 프레임워크 | Strands Agents |
| LLM 모델 | Claude Haiku 4.5 |
| 메모리 전략 | 리플렉션 설정이 포함된 에피소딕 메모리 |
| 복잡도 | 중급 |

## 사전 요구사항

- Python 3.10+
- AgentCore 메모리 권한이 있는 AWS 자격 증명
- AgentCore 서비스에 대한 접근 권한

## 1단계: 종속성 설치 및 설정

In [ ]:
%pip install -qr requirements.txt

In [ ]:
import json
import logging
import sys
import uuid
from datetime import datetime, timezone
from typing import List, Dict, Any
from pprint import pprint

# 로깅 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("debugging-assistant")

# boto3 임포트 (컨트롤 플레인 및 데이터 플레인 작업용)
import boto3

# Strands Agent 프레임워크 임포트
from strands import Agent, tool

logger.info("모든 종속성이 성공적으로 임포트되었습니다")

In [ ]:
import os
# 설정
REGION = os.getenv('AWS_REGION', 'us-west-2')
# 세션 식별자
ACTOR_ID = "developer"

logger.info(f"리전 설정: {REGION}")
logger.info(f"Actor ID: {ACTOR_ID}")

## 2단계: 에피소딕 전략으로 메모리 생성

**리플렉션 설정**이 포함된 **에피소딕 메모리 전략**으로 설정된 메모리 리소스를 생성합니다. 이를 통해 다음이 가능합니다:
- 완전한 디버깅 세션 에피소드 저장
- 여러 에피소드에서 리플렉션 인사이트 자동 생성

In [ ]:
# 컨트롤 플레인 및 데이터 플레인 작업을 위한 boto3 클라이언트 초기화
client = boto3.client(
    'bedrock-agentcore',
    region_name=REGION,
)
memory_client = boto3.client(
  'bedrock-agentcore-control',
   region_name=REGION,
)

In [ ]:
# 리플렉션이 포함된 에피소딕 메모리 전략을 딕셔너리로 정의
memory_name = "DebugAssistantEpisodic"

# 에피소딕 메모리는 customMemoryStrategy로 구현됩니다
episodic_strategy = {
    "episodicMemoryStrategy": {
      "name": "DebuggingEpisodeExtractor",
      "description": "Creates debugging session episodes with reflections per actor",
      "namespaces": [
        "debugging/{actorId}/sessions/{sessionId}/"
      ],
      "reflectionConfiguration": {
        "namespaces": [
          "debugging/{actorId}/" # 에피소딕 메모리 네임스페이스의 정확한 접두사여야 합니다
        ]
      }
    }
}
logger.info(f"전략 설정 완료: {episodic_strategy['episodicMemoryStrategy']['name']}")
logger.info(f"에피소드 네임스페이스: {episodic_strategy['episodicMemoryStrategy']['namespaces'][0]}")
logger.info(f"리플렉션 네임스페이스: {episodic_strategy['episodicMemoryStrategy']['reflectionConfiguration']['namespaces'][0]}")

In [ ]:
# 메모리 가져오기 또는 생성
try:
    # 먼저 기존 메모리 검색 시도
    list_response = memory_client.list_memories(maxResults=100)
    memory_id = None
    for mem in list_response.get('memories', []):
        detail = memory_client.get_memory(memoryId=mem['id'])
        if detail['memory'].get('name') == memory_name:
            memory_id = mem['id']
            logger.info(f"기존 메모리 사용: {memory_id}")
            break
    
    # 존재하지 않으면 생성
    if not memory_id:
        logger.info(f"새 메모리 생성: {memory_name}")
        response = memory_client.create_memory(
            name=memory_name,
            description="Episodic memory for debugging assistant with reflections",
            eventExpiryDuration=90,
            memoryStrategies=[episodic_strategy],
            clientToken=str(uuid.uuid4())
        )
        memory_id = response['memory']['id']
        logger.info(f"메모리 생성 완료: {memory_id}")
        
        # ACTIVE 상태까지 대기
        import time
        for _ in range(30):
            status = memory_client.get_memory(memoryId=memory_id)['memory']['status']
            if status == 'ACTIVE':
                logger.info("메모리가 ACTIVE 상태입니다")
                break
            time.sleep(10)
            
except Exception as e:
    logger.error(f"메모리 가져오기/생성 실패: {e}")
    raise

## 4단계: 과거 디버깅 세션으로 메모리 하이드레이션

과거 디버깅 세션을 에피소딕 메모리에 로드합니다. 각 세션은 완전한 디버깅 워크플로를 나타냅니다.

In [ ]:
import os
import glob

# 모든 세션 데이터 파일 로드
data_dir = "./data"
session_files = sorted(glob.glob(f"{data_dir}/*.json"))

logger.info(f"하이드레이션할 세션 파일 {len(session_files)}개 발견")

# 각 세션 하이드레이션
for session_file in session_files:
    session_name = os.path.basename(session_file).replace('.json', '')
    session_id = f"{session_name}_{datetime.now().strftime('%Y%m%d%H%M%S')}"
    
    logger.info(f"세션 하이드레이션 중: {session_name}")
    
    # 대화 데이터 로드
    with open(session_file, 'r') as f:
        conversation = json.load(f)
    
    # 페이로드 형식으로 변환
    payload = []
    for turn in conversation:
        conv_data = turn['conversational']
        payload.append({
            'conversational': {
                'content': {'text': conv_data['content']['text']},
                'role': conv_data['role']
            }
        })
    
    # boto3를 직접 사용하여 이벤트 생성
    event_timestamp = datetime.now(timezone.utc)
    result = client.create_event(
        memoryId=memory_id,
        actorId=ACTOR_ID,
        sessionId=session_id,
        eventTimestamp=event_timestamp,
        payload=payload
    )

    logger.info(f"   {len(payload)}개의 턴 저장 완료 - Event ID: {result['event']['eventId']}")

logger.info(f"{len(session_files)}개의 디버깅 세션 하이드레이션 완료")

In [ ]:
### 메모리 레코드 목록을 조회하여 LTM으로 추출되었는지 확인
import time
import pprint
reflection_namespace = f"debugging/{ACTOR_ID}/"
# time.sleep(60)
# boto3 클라이언트를 직접 사용하여 메모리 레코드 검색
response = client.list_memory_records(
    memoryId=memory_id,
    namespace=reflection_namespace,
    maxResults=20
)
memories = response.get('memoryRecordSummaries', [])
logger.info(f"   {len(memories)}개의 메모리를 발견했습니다")
if memories:
    pprint.pp(json.loads(memories[0]["content"]["text"]))

In [ ]:
# 리플렉션과 에피소드가 생성되었는지 확인
import pprint
# boto3 클라이언트를 직접 사용하여 메모리 레코드 검색
response = client.retrieve_memory_records(
memoryId=memory_id,
namespace=f"debugging/{ACTOR_ID}/",
searchCriteria={
    'searchQuery': "memory leaks",
    "metadataFilters":[{"left": {"metadataKey": "x-amz-agentcore-memory-recordType"},
        "operator": "EQUALS_TO",
        "right": {"metadataValue": {"stringValue": "REFLECTION"}}
        }],          
    'topK': 10
},
    maxResults=20
)

reflections = response.get('memoryRecordSummaries', [])
logger.info(f"   {len(reflections)}개의 관련 리플렉션을 발견했습니다")
if reflections:
    for reflection in reflections:
        reflection_json = json.loads(reflection['content']['text'])
        pprint.pp(reflection_json)

## 5단계: 메모리 검색 도구 생성

에이전트를 위한 두 가지 전문 도구를 생성합니다:
1. **retrieve_process**: 상세한 단계별 프로세스를 위한 완전한 에피소드 추적을 검색합니다
2. **retrieve_reflection_knowledge**: 여러 에피소드에서 합성된 인사이트와 패턴을 검색합니다

In [ ]:
def count_tokens(text: str) -> int:
    """텍스트 문자열의 대략적인 토큰 수를 계산합니다."""
    return len(text)


def linearize_episodes(episodes: List[Dict], include_steps: bool = True,
                       include_reflections: bool = True) -> str:
    """에피소드 데이터를 사람이 읽을 수 있는 형식으로 직렬화합니다."""
    if not episodes:
        return "No relevant episodes found."

    output = []
    for idx, episode in enumerate(episodes, 1):
        content = episode.get('content', {})
        
        # text 필드에서 JSON 파싱
        if 'text' in content:
            try:
                episode_data = json.loads(content['text'])
            except json.JSONDecodeError:
                output.append(f"Episode {idx}: Unable to parse content\n")
                continue
        else:
            output.append(f"Episode {idx}: No content available\n")
            continue
        
        output.append(f"{'='*80}\nEpisode {idx}\n{'='*80}")
        output.append(f"**Situation:** {episode_data.get('situation', 'N/A')}")
        output.append(f"**Intent:** {episode_data.get('intent', 'N/A')}")
        output.append(f"**Assessment:** {episode_data.get('assessment', 'N/A')}\n")
        
        if include_steps:
            turns = episode_data.get('turns', [])
            if turns:
                output.append("**Debugging Steps:**")
                for turn_idx, turn in enumerate(turns, 1):
                    output.append(f"\nStep {turn_idx}:")
                    output.append(f"  Situation: {turn.get('situation', 'N/A')}")
                    output.append(f"  Action: {turn.get('action', 'N/A')}")
                    output.append(f"  Thought: {turn.get('thought', 'N/A')}")
        
        if include_reflections:
            reflection = episode_data.get('reflection')
            if reflection:
                output.append(f"\n**Reflection:** {reflection}\n")
    
    result = "\n".join(output)
    logger.info(f"   에피소드 토큰: {count_tokens(result)}")
    return result


def linearize_reflections(reflections: List[Dict]) -> str:
    """리플렉션 지식을 사람이 읽을 수 있는 형식으로 직렬화합니다."""
    if not reflections:
        return "No reflection knowledge found."

    output = []
    for idx, reflection in enumerate(reflections, 1):
        content = reflection.get('content', {})
        score = reflection.get('score', 0)
        
        # text 필드에서 JSON 파싱
        if 'text' in content:
            try:
                reflection_data = json.loads(content['text'])
            except json.JSONDecodeError:
                continue
        else:
            continue

        output.append(f"{'='*80}\nReflection {idx} (Relevance: {score:.2f})\n{'='*80}")
        output.append(f"**Title:** {reflection_data.get('title', 'Untitled')}")
        output.append(f"**Use Cases:** {reflection_data.get('use_cases', 'N/A')}")
        output.append(f"**Hints:** {reflection_data.get('hints', 'N/A')}\n")
    
    result = "\n".join(output)
    logger.info(f"   리플렉션 토큰: {count_tokens(result)}")
    return result


logger.info("직렬화 헬퍼 함수 생성 완료")

In [ ]:
# 에이전트용 메모리 검색 도구 생성

@tool
def retrieve_process(task: str, include_steps: bool = True) -> str:
    """
    Retrieve example processes to help solve the given task. Returns complete debugging
    episodes with configurable detail level.
    
    Use include_steps parameter to control verbosity:
    - Set include_steps=True when user asks for "exact steps", "full details", "how did we",
      "what steps did we take", or needs complete procedural information
    - Set include_steps=False for pattern/best practice queries where high-level context
      (situation, intent, assessment) is sufficient without step-by-step details

    Args:
        task: The task to solve that requires example processes
        include_steps: Whether to include detailed step-by-step turns (default: True)

    Returns:
        Formatted debugging episodes with optional detailed steps
    """
    logger.info(f"작업에 대한 프로세스 검색 중: {task} (include_steps={include_steps})")
    
    try:
        # 에피소드 네임스페이스에서 검색
        namespace = f"debugging/{ACTOR_ID}/sessions/{session_id}/"
        
        # boto3 클라이언트를 직접 사용하여 메모리 레코드 검색
        response = client.retrieve_memory_records(
            memoryId=memory_id,
            namespace=namespace,
            searchCriteria={
                'searchQuery': task,
                'topK': 3
            },
            maxResults=20
        )
        
        episodes = response.get('memoryRecordSummaries', [])
        logger.info(f"   {len(episodes)}개의 관련 에피소드를 발견했습니다")
        
        # 설정 가능한 세부 수준으로 직렬화
        return linearize_episodes(episodes, include_steps=include_steps, include_reflections=True)
        
    except Exception as e:
        logger.error(f"프로세스 검색 오류: {e}")
        return f"Error retrieving processes: {str(e)}"


@tool
def retrieve_reflection_knowledge(task: str, k: int = 5) -> str:
    """
    Retrieve synthesized reflection knowledge from past agent experiences. Each knowledge 
    entry contains: (1) a descriptive title, (2) specific use cases for when to apply it, 
    and (3) actionable hints including best practices from successful episodes and common 
    pitfalls to avoid from failed episodes. Use this to get strategic guidance and patterns
    for similar tasks.

    Args:
        task: The current task to get strategic guidance for
        k: Number of reflection entries to retrieve (default: 5)

    Returns:
        Synthesized reflection knowledge from past debugging experiences
    """
    logger.info(f"작업에 대한 리플렉션 지식 검색 중: {task}")
    
    try:
        # 리플렉션 네임스페이스에서 검색
        namespace = f"debugging/{ACTOR_ID}/"
        
        # boto3 클라이언트를 직접 사용하여 메모리 레코드 검색
        response = client.retrieve_memory_records(
            memoryId=memory_id,
            namespace=namespace,
            searchCriteria={
                'searchQuery': "memory leaks",
                "metadataFilters":[{"left":{"metadataKey": "x-amz-agentcore-memory-recordType"},
                                    "operator": "EQUALS_TO",
                                    "right": {"metadataValue": {"stringValue": "REFLECTION"}}
                                    }],          
                'topK': k
            },
            maxResults=20
        )
        
        reflections = response.get('memoryRecordSummaries', [])
        logger.info(f"   {len(reflections)}개의 관련 리플렉션 인사이트를 발견했습니다")
        
        # 리플렉션 직렬화
        return linearize_reflections(reflections)
        
    except Exception as e:
        logger.error(f"리플렉션 검색 오류: {e}")
        return f"Error retrieving reflections: {str(e)}"


logger.info("메모리 검색 도구 생성 완료")

## 6단계: 디버깅 어시스턴트 에이전트 생성

이제 메모리 검색 도구가 장착된 Strands 에이전트를 생성합니다.

In [ ]:
# 디버깅 어시스턴트 에이전트 생성
debugging_agent = Agent(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[retrieve_process, retrieve_reflection_knowledge],
    system_prompt="""You are an expert Debugging Assistant with access to episodic memory.

Your capabilities:
- Retrieve past debugging episodes with complete step-by-step processes
- Access synthesized reflection knowledge showing patterns and best practices
- Provide guidance based on successful debugging experiences
- Warn about common pitfalls observed in past failures

When helping users:
1. Use retrieve_reflection_knowledge for strategic guidance, patterns, and high-level advice
2. Use retrieve_process when users need exact steps or want to recall what was done in a specific session
3. Synthesize insights from memory with your own reasoning
4. Be specific and actionable in your recommendations

Always explain your reasoning and cite relevant past experiences when available."""
)

logger.info("디버깅 어시스턴트 에이전트 생성 완료")

## 7단계: 디버깅 어시스턴트 테스트

에이전트가 에피소딕 메모리와 리플렉션을 어떻게 활용하는지 다양한 시나리오를 테스트합니다.

### 테스트 1: 전략적 가이드 쿼리 (리플렉션 지식)

In [ ]:
# 테스트 1: 메모리 문제에 대한 전략적 가이드 요청
# 대규모 데이터셋 처리 시 애플리케이션 메모리 부족 문제. 무엇을 확인해야 하나요?
query1 = "My application is running out of memory when processing large datasets. What should I look for?"

logger.info(f"\n{'='*80}")
logger.info(f"테스트 1: 메모리 문제 가이드")
logger.info(f"{'='*80}")
logger.info(f"쿼리: {query1}\n")

response1 = debugging_agent(query1)

print("\n" + "="*80)
print("에이전트 응답:")
print("="*80)
print(response1)

### 테스트 2: 특정 프로세스 세부사항 쿼리

In [ ]:
# 테스트 2: 특정 디버깅 프로세스 요청
# 외부 API 호출의 타임아웃 문제를 디버깅하는 정확한 단계를 보여주세요.
query2 = "Show me the exact steps for debugging a timeout issue with external API calls."

logger.info(f"\n{'='*80}")
logger.info(f"테스트 2: API 타임아웃 디버깅 프로세스")
logger.info(f"{'='*80}")
logger.info(f"쿼리: {query2}\n")

response2 = debugging_agent(query2)

print("\n" + "="*80)
print("에이전트 응답:")
print("="*80)
print(response2)

### 테스트 3: 패턴 인식 쿼리

In [ ]:
# 테스트 3: 동시성 문제에 대한 패턴 및 모범 사례 요청
# 멀티스레드 애플리케이션에서 레이스 컨디션을 처리하는 일반적인 패턴과 모범 사례는 무엇인가요?
query3 = "What are common patterns and best practices for handling race conditions in multi-threaded applications?"

logger.info(f"\n{'='*80}")
logger.info(f"테스트 3: 레이스 컨디션 패턴")
logger.info(f"{'='*80}")
logger.info(f"쿼리: {query3}\n")

response3 = debugging_agent(query3)

print("\n" + "="*80)
print("에이전트 응답:")
print("="*80)
print(response3)

### 테스트 4: 특정 세션 회상

In [ ]:
# 테스트 4: 메모리 누수 세션에서 수행한 작업 회상
# 메모리 누수 문제를 겪었을 때 어떤 디버깅 단계를 수행했나요? 전체 세부사항이 필요합니다.
query4 = "What debugging steps did we take when we encountered the memory leak issue? I need the full details."

logger.info(f"\n{'='*80}")
logger.info(f"테스트 4: 메모리 누수 세션 회상")
logger.info(f"{'='*80}")
logger.info(f"쿼리: {query4}\n")

response4 = debugging_agent(query4)

print("\n" + "="*80)
print("에이전트 응답:")
print("="*80)
print(response4)

## 8단계: 직접 메모리 검사

에피소딕 메모리와 리플렉션에 저장된 내용을 직접 검사합니다.

In [ ]:
# boto3를 사용하여 에피소드 직접 검사
logger.info("" + "="*80)
logger.info("직접 에피소드 검사")
logger.info("="*80)

# boto3를 직접 사용하여 에피소드 검색
namespace = f"debugging/{ACTOR_ID}/sessions/"
response = client.retrieve_memory_records(
    memoryId=memory_id,
    namespace=namespace,
    searchCriteria={
        'searchQuery': 'debugging',
        'topK': 2
    },
    maxResults=10
)

episodes = response.get('memoryRecordSummaries', [])

print(f"메모리에서 {len(episodes)}개의 에피소드를 발견했습니다:")
for idx, episode in enumerate(episodes, 1):
    print(f"에피소드 {idx}:")
    pprint.pp(episode, depth=2, width=100)
    print("-" * 80)

In [ ]:
import pprint
response = client.retrieve_memory_records(
memoryId=memory_id,
namespace=reflection_namespace,
searchCriteria={
    'searchQuery': "memory leaks",
    "metadataFilters":[{
        "left":{"metadataKey": "x-amz-agentcore-memory-recordType"},
        "operator": "EQUALS_TO",
        "right": {"metadataValue": {"stringValue": "REFLECTION"}}
                        }],          
    'topK': 10
},
    maxResults=20
)

reflections = response.get('memoryRecordSummaries', [])
logger.info(f"   {len(reflections)}개의 관련 리플렉션을 발견했습니다")
if reflections:
    for reflection in reflections:
        reflection_json = json.loads(reflection['content']['text'])
        pprint.pp(reflection_json)

## 요약

### 달성한 내용

- boto3를 사용하여 리플렉션 설정이 포함된 에피소딕 메모리 생성

- 과거 디버깅 세션으로 메모리 하이드레이션

- 에피소드 및 리플렉션을 위한 전문 검색 도구 구축

- Strands 프레임워크를 사용하여 지능형 디버깅 어시스턴트 생성

- 전략적 가이드 검색 vs. 상세 프로세스 검색 시연

### 주요 시사점

1. **에피소딕 메모리**는 시간적 컨텍스트와 함께 완전한 상호작용 시퀀스를 보존합니다
2. **리플렉션**은 여러 에피소드에서 패턴과 인사이트를 자동으로 합성합니다
3. **직렬화**는 구조화된 데이터를 LLM 소비에 최적화된 형식으로 변환합니다
4. **도구 선택**이 중요합니다: 전략에는 리플렉션을, 상세 단계에는 에피소드를 사용합니다
5. **Boto3 직접 접근**은 Genesis Memory API 작업에 대한 완전한 제어를 제공합니다

### 이 패턴을 사용하는 경우

- 과거 티켓 해결에서 학습하는 **기술 지원 시스템**
- 성공적인 진단 절차를 기억하는 **문제 해결 어시스턴트**
- 지식 전달을 위해 전문가 워크플로를 캡처하는 **교육 시스템**
- 과거 결과 분석이 더 나은 실행을 이끄는 **프로세스 개선** 시나리오

## 정리 (선택 사항)

완료 후 메모리 리소스를 삭제하려면 주석을 해제하세요.

In [ ]:
# boto3를 사용하여 메모리 리소스를 삭제하려면 주석을 해제하세요
# try:
#     client.delete_memory(memoryId=memory_id, clientToken=str(uuid.uuid4()))
#     logger.info(f"메모리 삭제 완료: {memory_id}")
# except Exception as e:
#     logger.error(f"메모리 삭제 오류: {e}")